# Baseline Model 

This notebook contains the baseline model, a model with minimal preprocessing, no feature engineering at all. 

The goal of this notebook is to understand how good are the raw features without feature engineering, so that it can set a yardstick to evaluate future feature engineering.

In [15]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

In [16]:
import numpy as np
import pandas as pd

from src.data import load_train_data

In [17]:
data = load_train_data()

In [18]:
# define target, X, y
TARGET = 'SalePrice'

y = np.log1p(data[TARGET])
X = data.drop(columns=[TARGET]) # everything less the target


In [19]:
# Feature groups, reusing from EDA
nominal_cols = ['MSZoning','MSSubClass', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType','Foundation', 'Heating','CentralAir', 'Electrical','Functional',
       'GarageType', 'GarageFinish','PavedDrive','MiscFeature',
       'SaleType', 'SaleCondition','Fence'
]

ordinal_cols = ['OverallQual','OverallCond','ExterQual', 'ExterCond',
                'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                'HeatingQC','KitchenQual','FireplaceQu','GarageQual',
                'GarageCond','PoolQC'

]

categorical_cols = nominal_cols + ordinal_cols

In [20]:
# due to different nature of missingness, the way these missing attributes are handled will also be different

# classify numerical attributes with missing values 
num_structured_missing = ['MasVnrArea','GarageYrBlt']
num_unstruc_missing = ['LotFrontage']

## processing pipelines

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

In [22]:
# for numerical attributes with structured missingness, value 0 will be imputed 
num_struc_missing_pipeline = Pipeline([
  ('imputer',
   SimpleImputer(strategy='constant',
   fill_value = 0)),
  ('scaler',StandardScaler())
])


# for numerical attributes with true missingness, the median value will be imputed 
num_unstruc_missing_pipeline = Pipeline([
  ('imputer',
   SimpleImputer(strategy='median')),
   ('scaler',StandardScaler())
])

other_numerical_cols = [
  col for col in data.columns
  if col not in nominal_cols + ordinal_cols + ["SalePrice"] + num_structured_missing + num_unstruc_missing
]

For categorical attributes, I will use the most frequent value to impute missing values, previously in EDA I identified some categorical attributes with structured missingness indeed, but since this notebook focuses on baseline models, encoding them explicitly will be delayed into later stages of modeling

For numerical attributes, imputing all missing values with the median will incorrectly assign some quantity to a house with no such feature, but for categorical attributes, since one-hot encoding will be used to all categorical attributes, and one-hot encoding treats missing values as a seperate category implicitly, explicitly separating structured vs unstructured missingness does not add much value at this stage

In [23]:
# categorical attributes
categorcal_pipeline = Pipeline([
  ('imputer',
   SimpleImputer(strategy='most_frequent')),
   ('onehot',OneHotEncoder(handle_unknown='ignore'))
])

In [24]:
# column transformer
preprocessor = ColumnTransformer([
  ('num_zero', num_struc_missing_pipeline,
   num_structured_missing),
   ('num_median', num_unstruc_missing_pipeline,
    num_unstruc_missing),
    ('cat', categorcal_pipeline,
    categorical_cols )
])

## Model

I will use a linear model for the baseline. 

A linear model is  interpretable, its coefficients reflects feature influence, and it allows me to clearly evaluate if feature engineering and  the use of non-linear models are worthwhile 

I will use Ridge to regularise the linear regression, the dataset contains clusters of related attributes, without regularisation, the solution might be overly complex and does not generalise well

Ridge is preferred over Lasso, because Lasso might set the coefficients as 0 for some correlated features, while Ridge preserves them

This model will become a stable benchmark before using more complex models and making more decisions

The regularisation strength is set to be 1, a moderate strength

In [25]:
model = Ridge(alpha=1.0)

In [26]:
# Full pipeline
pipeline = Pipeline([
  ('preprocess',preprocessor),
  ('model',model)
])

I did not do a train_test_split because I will use K-fold cross-validation, it is a more reliable estimate of generalisation performance.

In [27]:
scores = cross_val_score(
  pipeline, X, y,
  scoring= 'neg_root_mean_squared_error',
  cv = 5
)

rmse = -scores.mean()
print(rmse)

0.1719495706843155


The resulting RMSE (~0.17 in log space) will serve as a reference score for evaluating complex models and feature engineering in the future